<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [155]:
import numpy as np
from numpy.polynomial import Polynomial
import re

def coef_amort(Mp_percent):
  # calcula o coeficiente de amortecimento a partir do overshoot em porcentagem
  log_mp = np.log(Mp_percent/100)
  return log_mp/np.sqrt(np.pi**2 + log_mp**2)

def freq_natural(ksi, tss, tss_percent):
  # calcula a frequência natural a partir do ksi, tss e a acomodação
  if(tss_percent == 5):
    constante = 3
  elif(tss_percent == 2):
    constante = 5
  return constante/(tss*ksi)

def sigma_angulos(s, pontos):
  sigma = 0
  for i in range(len(pontos)):
    sigma += np.rad2deg(np.atan2((s - pontos[i]).imag, (s - pontos[i]).real))
  return sigma

def phi_modulos(s, pontos):
  phi = 1
  for i in range(len(pontos)):
    phi *= np.abs((s - pontos[i]))
  return phi

In [157]:
print("Coeficientes do numerador de G(s): ")
s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s))

print("Os coeficentes digitados foram: ")
for i in range(len(coef_arr)):
  if(i == len(coef_arr)-1):
    print(coef_arr[i])
  else:
    print(coef_arr[i]+', ', end = '')

#print(np.sum([1, 2, 1]))

TypeError: 'numpy.int64' object is not callable

In [145]:
print("Coeficientes do numerador de G(s): "); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s))

coefGNum = np.array(coef_arr); coefGDen = np.array([1, 4, 4, 0])
g_num = Polynomial(np.flip(coefGNum))
g_den = Polynomial(np.flip(coefGDen))
print("G(x) = (" + str(g_num) + ")/(" + str(g_den) + ")")

coefHNum = np.array([1]); coefHDen = np.array([1])
h_num = Polynomial(np.flip(coefHNum))
h_den = Polynomial(np.flip(coefHDen))
print("\nH(x) = (" + str(h_num) + ")/(" + str(h_den) + ")")

Coeficientes do numerador de G(s): 
4 16


ValueError: Coefficient arrays have no common type

In [108]:
Mp_percent = 10; tss = 4; tss_percent = 5

ksi = np.abs(coef_amort(Mp_percent)); ksi = np.round(ksi,4)
wn = freq_natural(ksi,tss,tss_percent); wn = np.round(wn,4)
print("Coeficiente de amortecimento: " + str(ksi))
print("Frequência natural: " + str(wn) + " rad/s")

Coeficiente de amortecimento: 0.5912
Frequência natural: 1.2686 rad/s


In [56]:
wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
print("Frequência amortecida: " + str(wd))

s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)

Frequência amortecida: 1.0232


In [57]:
print("s1: " + str(s1))
print("s2: " + str(s2))

s1: (-0.75+1.0232j)
s2: (-0.75-1.0232j)


In [115]:
# Controlador PD

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = kd*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 8.6602
Ganho do controlador projetado: 0.0305

Ganho proporcional do controlador: 0.2637
Ganho derivativo do controlador: 0.0305


In [38]:
# Controlador PI

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = kc; ki = kp*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))

Zero(s) do controlador projetado: 3.4286
Ganho do controlador projetado: 7.0

Ganho proporcional do controlador: 7.0
Ganho integrativo do controlador: 24.0


In [53]:
# Controlador PID

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = (sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180)/2
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = 2*z*kd; ki = kd*np.power(z,2);
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 2.049
Ganho do controlador projetado: 3.137

Ganho proporcional do controlador: 12.8558
Ganho integrativo do controlador: 13.1711
Ganho derivativo do controlador: 3.137
